# Ingestion : scraper les comptes annuels sur la Centrale des bilans (NBB/CBSO)

Objectif : à partir des numéros d'entreprise déjà en base, aller chercher les comptes annuels publiés sur le site de la Centrale des bilans de la Banque Nationale de Belgique

## 1. Découverte du site

Avant d'écrire la moindre ligne de code, allez consulter le site normalement, dans un navigateur : https://consult.cbso.nbb.be/consult-enterprise/0693810613

C'est la fiche d'une entreprise réelle, remplacez le numéro par n'importe quel numéro BCE (10 chiffres, sans points) pour voir une autre entreprise. Vous devriez voir la liste des comptes annuels déposés, année par année.

**À noter avant de continuer :**

- Depuis l'exercice comptable **2021**, les comptes sont disponibles au format **CSV** (en plus du PDF). Avant 2021, seul le PDF existe. C'est pourquoi on se concentre sur le CSV et sur les années récentes dans cet exercice, le PDF reste possible à télécharger en plus si vous voulez aller plus loin.(OCR)
- Cette page HTML n'est pas elle-même la source de données, elle appelle une API JSON, que vous allez interroger directement dans la suite.

## 2. Premier appel : lister les dépôts d'une entreprise

**Endpoint** : `https://consult.cbso.nbb.be/api/rs-consult/published-deposits`

**Paramètres de requête (query params)** à envoyer :

- `enterpriseNumber` : le numéro BCE, SANS points (ex. `0693810613`, pas `0693.810.613`)
- `page` : numéro de page, en partant de `0`
- `size` : taille de page (ex. `50`)
- `sort` : à envoyer comme une liste, avec DEUX valeurs (`periodEndDate,desc` et `depositDate,desc`) -- avec la librairie `requests`, un paramètre dont la valeur est une liste Python est automatiquement répété deux fois dans l'URL, ce qui correspond au format attendu par cette API

**Pagination** : la réponse JSON contient un champ `content` (la liste des dépôts de cette page) et un champ booléen `last`. Continuez à demander la page suivante tant que `last` vaut `false`. Attention : il n'y a PAS de champ `totalPages` malgré ce qu'on pourrait attendre d'une API paginée classique !!!

**En-têtes (headers)** à envoyer:

- `User-Agent` : une chaîne de navigateur réaliste (pas du n'importe quoi)
- `Accept`, `Accept-Language` (c'est mieux si c'est en francais)
- `Referer` : l'URL de la fiche entreprise correspondante (`https://consult.cbso.nbb.be/consult-enterprise/{numero}`)

**Session/cookies** : avant d'appeler l'API, faites d'abord une requête GET normale vers la fiche HTML de l'entreprise (`consult-enterprise/{numero}`) pour récupérer les cookies que le site pose sur un chargement de page classique. Réutilisez ensuite ces cookies pour l'appel API 


Chaque dépôt renvoyé contient au moins : `id` (identifiant du dépôt), `periodEndDateYear`, `language`, `modelName`

In [1]:
!pip install requests[socks] pymongo hdfs stem

zsh:1: no matches found: requests[socks]


In [2]:
import time
import requests
from typing import Optional  # <--- Ajout de l'import

HEADERS_BASE = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7",
}

def init_session(enterprise_number: str) -> requests.Session:
    """Initialise une session HTTP et récupère les cookies via la fiche entreprise[cite: 1]."""
    session = requests.Session()
    session.headers.update(HEADERS_BASE)
    referer_url = f"https://consult.cbso.nbb.be/consult-enterprise/{enterprise_number}"
    session.headers["Referer"] = referer_url
    
    # Premier appel GET sur la fiche HTML pour poser les cookies de session[cite: 1]
    resp = session.get(referer_url, timeout=10)
    resp.raise_for_status()
    return session

def get_published_deposits(session: requests.Session, enterprise_number: str) -> list:
    """Interroge l'API JSON et gère la pagination pour lister tous les dépôts d'une entreprise[cite: 1]."""
    url = "https://consult.cbso.nbb.be/api/rs-consult/published-deposits"
    deposits = []
    page = 0
    size = 50
    
    while True:
        params = [
            ("enterpriseNumber", enterprise_number),
            ("page", str(page)),
            ("size", str(size)),
            ("sort", "periodEndDate,desc"),
            ("sort", "depositDate,desc")
        ]
        
        resp = session.get(url, params=params, timeout=15)
        
        if resp.status_code == 429:
            raise requests.exceptions.HTTPError("429 Too Many Requests", response=resp)
            
        resp.raise_for_status()
        data = resp.json()
        
        content = data.get("content", [])
        deposits.extend(content)
        
        # Arrêter si 'last' est True (API sans totalPages)[cite: 1]
        if data.get("last", True):
            break
            
        page += 1
        
    return deposits

# --- MODIFICATION ICI : Optional[bytes] au lieu de bytes | None ---
def download_csv_deposit(session: requests.Session, deposit_id: str) -> Optional[bytes]:
    """Télécharge le fichier CSV lié à un dépôt donné[cite: 1]."""
    url = f"https://consult.cbso.nbb.be/api/external/broker/public/deposits/consult/csv/{deposit_id}"
    
    resp = session.get(url, timeout=15)
    
    # 404 / 500 : Pas de CSV pour ce dépôt (comportement attendu)[cite: 1]
    if resp.status_code in (404, 500):
        return None
        
    # 502 / 503 : Souci serveur temporaire (à réessayer)[cite: 1]
    if resp.status_code in (502, 503):
        raise requests.exceptions.HTTPError(f"Erreur temporaire {resp.status_code}", response=resp)
        
    # 429 : Limite de requêtes atteinte[cite: 1]
    if resp.status_code == 429:
        raise requests.exceptions.HTTPError("429 Too Many Requests", response=resp)
        
    resp.raise_for_status()
    
    content = resp.content
    # Moins de 100 octets = fichier quasiment vide (pas de fichier)[cite: 1]
    if len(content) < 100:
        return None
        
    return content

# Test rapide sur l'entreprise d'exemple
session_test = init_session("0693810613")
liste_depots = get_published_deposits(session_test, "0693810613")
print(f"Nombre de dépôts trouvés : {len(liste_depots)}")
if liste_depots:
    print("Exemple de dépôt :", liste_depots[0])

/Users/theo-dev/Dev/M2_IPSSI/M2_BIGDATA/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


HTTPError: 429 Too Many Requests

## 3. Télécharger un dépôt CSV

Une fois qu'on a l'`id` d'un dépôt (récupéré à l'étape précédente), le CSV se télécharge via :

`https://consult.cbso.nbb.be/api/external/broker/public/deposits/consult/csv/{id}`

Exemple concret (id réel) : https://consult.cbso.nbb.be/api/external/broker/public/deposits/consult/csv/3cf4404a-7ba0-11f1-92d9-1db02102d1ba

Points à gérer :

- Réutilisez la même session (mêmes cookies/en-têtes que pour l'étape 2).
- Un statut `404` ou `500` veut dire qu'il n'y a pas de CSV pour ce dépôt (normal, ne pas relancer)
- Un statut `502`/`503` est temporaire (le serveur a un souci passager), celui-là, il faut le réessayer plus tard, pas l'ignorer. (timeout ou bien skip)
- Une réponse `200` mais avec un contenu très court (quelques dizaines d'octets) correspond en général à un fichier vide, à traiter comme "pas de fichier" aussi.

## 4. Scraper en continu jusqu'au 429, et gérer le cooldown

Faites tourner vos appels en boucle sur plusieurs entreprises, jusqu'à obtenir un code `429 Too Many Requests`.

quand vous obtenez ce 429, affichez l'intégralité des en-têtes de la réponse (`dict(resp.headers)`). Certaines API renvoient un en-tête `Retry-After` qui indique précisément combien de secondes attendre avant de réessayer, c'est à vous de regarder si CBSO l'envoie.

- **Si l'en-tête est présent** : attendez exactement la durée qu'il indique avant de réessayer.
- **Si l'en-tête est absent** : repli sur un backoff exponentiel (attendre un peu, puis de plus en plus longtemps à chaque 429 consécutif, jusqu'à un plafond raisonnable comme 120 secondes)

In [ ]:
def handle_rate_limit(response: requests.Response, consecutive_429: int) -> int:
    """Gère le cooldown lors d'un code 429 en analysant les headers ou via backoff[cite: 1]."""
    headers = dict(response.headers)
    print(f"[HTTP 429] Headers de la réponse : {headers}")
    
    retry_after = response.headers.get("Retry-After")
    
    # 1. Si Retry-After est fourni par l'API[cite: 1]
    if retry_after and retry_after.isdigit():
        wait_time = int(retry_after)
        print(f"-> En-tête Retry-After détecté : pause de {wait_time}s[cite: 1]")
    # 2. Sinon, repli sur un backoff exponentiel plafonné à 120s[cite: 1]
    else:
        wait_time = min(120, 5 * (2 ** consecutive_429))
        print(f"-> Retry-After absent : pause exponential backoff de {wait_time}s[cite: 1]")
        
    time.sleep(wait_time)
    return wait_time

## 5. Stocker les fichiers dans HDFS + suivi scrapping

**Stockage HDFS** : un dossier par entreprise, avec les CSV dedans, respectez la structure :

`/data/raw/{numero_entreprise}/cbso/csvs/{annee}.csv`

Avant d'écrire un fichier, vérifiez s'il existe déjà à ce chemin (pour ne pas retélécharger ce qu'on a déjà).

**Suivi des entreprises déjà traitées** : en plus de la vérification par fichier HDFS ci-dessus, tenez un petit fichier JSON simple qui note, pour chaque entreprise déjà passée en revue, si elle a été traitée, sauter directement les entreprises déjà vues

In [ ]:
import os
import json
from hdfs import InsecureClient

# Client HDFS (adaptez l'URL selon votre infrastructure)
hdfs_client = InsecureClient('http://localhost:9870', user='hdfs')
TRACKING_FILE_LOCAL = "scraped_enterprises.json"

def get_tracked_enterprises() -> set:
    """Récupère l'ensemble des entreprises déjà traitées via le fichier JSON local[cite: 1]."""
    if os.path.exists(TRACKING_FILE_LOCAL):
        with open(TRACKING_FILE_LOCAL, "r", encoding="utf-8") as f:
            return set(json.load(f))
    return set()

def update_tracked_enterprises(processed_set: set):
    """Met à jour le fichier JSON de suivi local[cite: 1]."""
    with open(TRACKING_FILE_LOCAL, "w", encoding="utf-8") as f:
        json.dump(list(processed_set), f, indent=2)

def save_csv_to_hdfs(enterprise_number: str, year: int, csv_content: bytes):
    """Enregistre le fichier CSV dans la structure HDFS exigée[cite: 1]."""
    hdfs_path = f"/data/raw/{enterprise_number}/cbso/csvs/{year}.csv"
    
    # Vérification d'existence avant écriture pour éviter le sur-téléchargement[cite: 1]
    if hdfs_client.status(hdfs_path, strict=False):
        print(f"Fichier déjà présent sur HDFS : {hdfs_path}[cite: 1]")
        return
        
    hdfs_client.write(hdfs_path, data=csv_content, overwrite=True)
    print(f"Sauvegardé sur HDFS : {hdfs_path}")

## 6. Passer par Tor : un premier exemple simple

**Service Docker** (`docker-compose.yml`) : un conteneur Tor avec l'image `dperson/torproxy` expose un proxy SOCKS5 sur le port `9050`, par exemple :

```yaml
  tor1:
    image: dperson/torproxy
    ports:
      - "9050:9050"
      - "9051:9051"
```

**Requête Python via Tor** : la librairie `requests` sait parler à un proxy SOCKS5 nativement, à condition d'installer `requests[socks]` (qui installe PySocks). Il suffit de configurer `session.proxies` avec une URL au format `socks5h://<hôte>:9050` (le `h` final est important : il dit à `requests` de résoudre les noms de domaine À TRAVERS Tor aussi, pas seulement le trafic).

Faites un test simple : une requête GET vers un service qui renvoie votre IP publique (par exemple un endpoint "what is my ip"), une fois SANS proxy, une fois AVEC le proxy Tor -- vous devriez voir deux IP différentes, ce qui confirme que le trafic passe bien par Tor.

In [ ]:
{"ip":"88.171.27.230"}
{"ip":"69.101.18.125"}

{'ip': '69.101.18.125'}

## 7. Plusieurs instances Tor, avec rotation

on ne veut pas exposer notre IP réelle, pour pouvoir continuer à utiliser le site à des fins de recherche sans risquer un blocage définitif de notre propre adresse. Faire tourner le trafic sur plusieurs identités Tor, et changer d'identité quand l'une d'elles se fait bloquer/limiter, permet de continuer à travailler sans jamais exposer la vraie IP de la machine.

créez 3 services Tor distincts dans le docker-compose (même image que l'étape 6, des noms différents, par ex. `tor1`/`tor2`/`tor3`, des ports différents sur l'hôte si vous voulez y accéder depuis votre machine, mais en interne au réseau docker, chacun écoute toujours sur 9050/9051).

Tor expose un "port de contrôle" (9051 par défaut) qui accepte une commande `SIGNAL NEWNYM` pour forcer la construction d'un nouveau circuit (donc une nouvelle IP de sortie) sur demande. Ce port demande une authentification (mot de passe)

Écrivez une petite classe ou fonction qui : garde une liste de vos 3 proxies, envoie les requêtes via le proxy courant, et sur un 429 (ou un blocage), envoie `SIGNAL NEWNYM` au proxy courant PUIS passe au proxy suivant de la liste avant de réessayer.

In [ ]:
from stem import Signal
from stem.control import Controller

class TorRotationManager:
    def __init__(self, proxy_list: list[dict]):
        """
        proxy_list attend un format du type :
        [
            {"socks": "socks5h://tor1:9050", "control_host": "tor1", "control_port": 9051, "password": "secret"},
            {"socks": "socks5h://tor2:9050", "control_host": "tor2", "control_port": 9051, "password": "secret"},
            {"socks": "socks5h://tor3:9050", "control_host": "tor3", "control_port": 9051, "password": "secret"}
        ][cite: 1]
        """
        self.proxies = proxy_list
        self.index = 0

    @property
    def current_proxy(self) -> dict:
        return self.proxies[self.index]

    def get_requests_proxies(self) -> dict:
        """Retourne la configuration au format attendu par requests (socks5h://)[cite: 1]."""
        url = self.current_proxy["socks"]
        return {"http": url, "https": url}

    def rotate(self):
        """Demande une nouvelle identité Tor (NEWNYM) et bascule vers le proxy suivant[cite: 1]."""
        current = self.current_proxy
        try:
            with Controller.from_port(address=current["control_host"], port=current["control_port"]) as controller:
                controller.authenticate(password=current.get("password", ""))
                controller.signal(Signal.NEWNYM)
                print(f"[Tor] Nouveaux circuits demandés pour {current['control_host']}[cite: 1]")
        except Exception as err:
            print(f"[Tor Alert] Connexion au port de contrôle échouée ({current['control_host']}): {err}")

        # Passer à l'instance Tor suivante[cite: 1]
        self.index = (self.index + 1) % len(self.proxies)
        print(f"[Tor] Nouveau proxy actif : {self.current_proxy['socks']}[cite: 1]")

## 9. Cibler intelligemment : échantillonnage stratifié par forme juridique

1. Récupérez, depuis MongoDB, la liste des valeurs distinctes du champ `JuridicalForm` présentes dans la collection `entreprise`.
2. Pour chaque valeur, tirez un échantillon (n>100) ALÉATOIRE d'une centaine d'entreprises ayant cette forme juridique 
3. Pour chaque entreprise de l'échantillon, faites juste l'appel de listage
4. Calculez, par forme juridique, le pourcentage d'entreprises SANS aucun dépôt.
5. Les formes juridiques dont ce pourcentage dépasse **95%** 

Le résultat de cette étape est une LISTE DE FORMES JURIDIQUES À EXCLURE, que vous réutiliserez à l'étape suivante pour filtrer les entreprises à traiter réellement.

In [ ]:
import requests
from pymongo import MongoClient

def find_excluded_juridical_forms(mongo_uri: str, db_name: str, sample_size: int = 100) -> list:
    """Exécute l'échantillonnage stratifié et filtre les formes juridiques > 95% sans dépôt."""
    client = MongoClient(mongo_uri)
    db = client[db_name]
    
    # 1. Obtenir les formes juridiques distinctes
    juridical_forms = db.entreprise.distinct("JuridicalForm")
    excluded_forms = []

    for j_form in juridical_forms:
        if not j_form:
            continue

        # 2. Échantillon aléatoire[cite: 1, 2, 3]
        sample = list(db.entreprise.aggregate([
            {"$match": {"JuridicalForm": j_form}},
            {"$sample": {"size": sample_size}}
        ]))

        if not sample:
            continue

        no_deposits_count = 0
        
        # 3. Tester le listage pour chaque entreprise de l'échantillon[cite: 1, 2, 3]
        for ent in sample:
            # Correction de la clé : 'EnterpriseNumber' avec E majuscule[cite: 3]
            raw_num = ent.get("EnterpriseNumber")
            if not raw_num:
                no_deposits_count += 1
                continue
                
            # Nettoyage des points pour l'API BNB (ex: "0200.065.765" -> "0200065765")[cite: 1, 2, 3]
            num_clean = str(raw_num).replace(".", "").strip()
            
            try:
                session = init_session(num_clean)
                deposits = get_published_deposits(session, num_clean)
                
                # Filtrer sur les comptes au format CSV (>= 2021)[cite: 1, 2, 3]
                recent = [d for d in deposits if d.get("periodEndDateYear", 0) >= 2021]
                if len(recent) == 0:
                    no_deposits_count += 1
            except Exception:
                no_deposits_count += 1

        # 4. Calcul du pourcentage d'entreprises sans dépôt[cite: 1, 2, 3]
        ratio_no_deposit = no_deposits_count / len(sample)
        print(f"Forme Juridique: {j_form} | Taux sans dépôt: {ratio_no_deposit * 100:.2f}%")

        # 5. Seuil d'exclusion > 95%[cite: 1, 2, 3]
        if ratio_no_deposit > 0.95:
            excluded_forms.append(j_form)

    print(f"\n[Filtre Stratifié] Liste des formes juridiques exclues : {excluded_forms}")
    return excluded_forms

## 10. Collection MongoDB de suivi du scraping

Créez une nouvelle collection, avec un document par entreprise CIBLE

- `enterpriseNumber`
- `juridicalForm` (pour pouvoir vérifier après coup que le filtre a bien été appliqué)
- `status` (`pending`, `done`, `error`)
- `lastScrapedAt` (timestamp du dernier passage)
- `documentsDownloaded` (nombre de CSV effectivement récupérés pour cette entreprise)

ne traiter que les entreprises dont la forme juridique n'est pas exclue ET dont le statut n'est pas déjà `done`. À chaque entreprise traitée, mettez à jour son document dans cette collection.

In [ ]:
import logging
from datetime import datetime
from pymongo import MongoClient
import requests

# Configuration du logger pour suivre le retour HTTP
logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger(__name__)

HEADERS_BASE = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8"
}

def run_cbso_scraping(
    mongo_uri: str, 
    db_name: str, 
    excluded_forms: list[str], 
    tor_mgr: TorRotationManager,
    limit: int = None  # <--- Ajout du paramètre de limite (ex: limit=10)
):
    """Pipeline final de scraping sécurisé avec suivi d'état dans MongoDB."""
    client = MongoClient(mongo_uri)
    db = client[db_name]
    tracking_col = db["scraping_status"]
    
    tracking_col.create_index("enterpriseNumber", unique=True)

    # 1. Sélection des cibles non exclues
    valid_enterprises = db.entreprise.find({"JuridicalForm": {"$nin": excluded_forms}})
    
    inserted_count = 0
    for ent in valid_enterprises:
        raw_num = ent.get("EnterpriseNumber")
        j_form = ent.get("JuridicalForm")
        
        if raw_num:
            num_clean = str(raw_num).replace(".", "").strip()
            tracking_col.update_one(
                {"enterpriseNumber": num_clean},
                {"$setOnInsert": {
                    "enterpriseNumber": num_clean,
                    "juridicalForm": j_form,
                    "status": "pending",
                    "lastScrapedAt": None,
                    "documentsDownloaded": 0
                }},
                upsert=True
            )
            inserted_count += 1

    logger.info(f"Cibles synchronisées dans scraping_status : {inserted_count}")

    # 2. Récupération des entreprises non encore finalisées
    pending_items = list(tracking_col.find({"status": {"$ne": "done"}}))
    logger.info(f"Total entreprises restant à traiter : {len(pending_items)}")

    if limit:
        logger.info(f"🧪 MODE TEST : Traitement limité aux {limit} premières entreprises.")

    # 3. Boucle de scraping robuste avec limiteur
    processed_count = 0

    for item in pending_items:
        # --- Contrôle de la limite ---
        if limit is not None and processed_count >= limit:
            logger.info(f"🛑 Limite de {limit} entreprise(s) atteinte ! Fin de la session de test.")
            break

        num = item["enterpriseNumber"]
        consecutive_429 = 0

        try:
            session = requests.Session()
            session.headers.update(HEADERS_BASE)
            session.proxies = tor_mgr.get_requests_proxies()
            
            referer_url = f"https://consult.cbso.nbb.be/consult-enterprise/{num}"
            session.headers["Referer"] = referer_url
            
            # Initialisation (récupération des cookies) + LOG HTTP
            start_time = datetime.now()
            resp_init = session.get(referer_url, timeout=10)
            elapsed = (datetime.now() - start_time).total_seconds()
            
            logger.info(
                f"HTTP Return | Status: {resp_init.status_code} | "
                f"Enterprise: {num} | Time: {elapsed:.2f}s"
            )
            
            # Gestion explicite du 429
            if resp_init.status_code == 429:
                raise requests.exceptions.HTTPError("429 Too Many Requests", response=resp_init)
                
            resp_init.raise_for_status()

            # Appel API de listage des dépôts
            deposits = get_published_deposits(session, num)
            csv_deposits = [d for d in deposits if d.get("periodEndDateYear", 0) >= 2021]
            
            downloaded = 0
            for dep in csv_deposits:
                dep_id = dep.get("id")
                year = dep.get("periodEndDateYear")
                
                if dep_id and year:
                    csv_bytes = download_csv_deposit(session, dep_id)
                    if csv_bytes:
                        save_csv_to_hdfs(num, year, csv_bytes)
                        downloaded += 1

            # Succès : Mise à jour dans MongoDB
            tracking_col.update_one(
                {"enterpriseNumber": num},
                {"$set": {
                    "status": "done",
                    "lastScrapedAt": datetime.utcnow(),
                    "documentsDownloaded": downloaded
                }}
            )

            # Incrémentation du compteur de succès/tentatives
            processed_count += 1

        except requests.exceptions.HTTPError as err:
            status_code = err.response.status_code if err.response is not None else None
            logger.warning(f"HTTP Error {status_code} | Enterprise: {num}")
            
            if status_code == 429:
                handle_rate_limit(err.response, consecutive_429)
                consecutive_429 += 1
                tor_mgr.rotate()
            else:
                tracking_col.update_one(
                    {"enterpriseNumber": num},
                    {"$set": {"status": "error", "lastScrapedAt": datetime.utcnow()}}
                )
                processed_count += 1
                
        except Exception as e:
            logger.error(f"Erreur globale sur l'entreprise {num}: {e}")
            tracking_col.update_one(
                {"enterpriseNumber": num},
                {"$set": {"status": "error", "lastScrapedAt": datetime.utcnow()}}
            )
            processed_count += 1

In [ ]:
!pip install PySocks

You should consider upgrading via the '/Users/theo-dev/Dev/M2_IPSSI/M2_BIGDATA/.venv/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
import requests
import sys

tor_proxies_config = [
    {"name": "tor1", "socks": "socks5h://127.0.0.1:9050"},
    {"name": "tor2", "socks": "socks5h://127.0.0.1:9052"},
    {"name": "tor3", "socks": "socks5h://127.0.0.1:9054"}
]

TEST_URL = "https://consult.cbso.nbb.be/consult-enterprise/0553461016"

# Headers pour simuler un vrai navigateur et éviter le HTTP 403
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8"
}

def verify_tor_connections():
    print("=== VÉRIFICATION DU RÉSEAU ET DES PROXIES TOR ===")
    all_ok = True

    for config in tor_proxies_config:
        proxy_name = config["name"]
        proxy_url = config["socks"]
        
        proxies = {"http": proxy_url, "https": proxy_url}

        try:
            # Timeout à 15s le temps que Tor stabilise la route
            response = requests.get(TEST_URL, proxies=proxies, headers=HEADERS, timeout=15)
            if response.status_code == 200:
                print(f"✓ [{proxy_name}] Connexion OK ! (Status: {response.status_code})")
            else:
                print(f"⚠ [{proxy_name}] Code HTTP inattendu : {response.status_code}")
                all_ok = False
        except Exception as e:
            print(f"✗ [{proxy_name}] ÉCHEC via {proxy_url} : {e}")
            all_ok = False

    print("=================================================")
    return all_ok

if not verify_tor_connections():
    print("🔴 ABANDON : Attends 10 secondes que les circuits Tor s'établissent puis relance.")
else:
    print("🟢 TOUT EST OK ! Tu peux lancer run_cbso_scraping().")

=== VÉRIFICATION DU RÉSEAU ET DES PROXIES TOR ===
✓ [tor1] Connexion OK ! (Status: 200)
✓ [tor2] Connexion OK ! (Status: 200)
✓ [tor3] Connexion OK ! (Status: 200)
🟢 TOUT EST OK ! Tu peux lancer run_cbso_scraping().


In [ ]:
import os
import time
import requests
from typing import Optional

def download_and_store_csv_stream(
    session: requests.Session, 
    enterprise_number: str, 
    dep_id: str, 
    year: int, 
    hdfs_client=None, 
    local_storage_dir: str = "./data/raw"
) -> bool:
    """
    Télécharge un fichier CSV et l'enregistre immédiatement sur le stockage 
    (Local et/ou HDFS) au fur et à mesure de la boucle.
    """
    local_path = os.path.join(local_storage_dir, enterprise_number, "cbso", "csvs", f"{year}.csv")
    hdfs_path = f"/data/raw/{enterprise_number}/cbso/csvs/{year}.csv"

    # 1. Vérification si le fichier existe déjà (évite de ré-interroger l'API)
    if os.path.exists(local_path):
        logger.info(f"⏩ Déjà téléchargé localement : {local_path}")
        return True

    url = f"https://consult.cbso.nbb.be/api/external/broker/public/deposits/consult/csv/{dep_id}"
    resp = session.get(url, timeout=15)
    
    if resp.status_code in (404, 500):
        return False
        
    if resp.status_code in (502, 503, 429):
        resp.raise_for_status()
        
    resp.raise_for_status()
    
    csv_bytes = resp.content
    if len(csv_bytes) < 100:
        return False

    # --- A. Stockage local immédiat sur disque ---
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    with open(local_path, "wb") as f:
        f.write(csv_bytes)
        f.flush()
        os.fsync(f.fileno()) # Écriture physique instantanée
    logger.info(f"💾 CSV sauvegardé immédiatement (Local) : {local_path}")

    # --- B. Stockage HDFS immédiat (si configuré) ---
    if hdfs_client:
        try:
            if not hdfs_client.status(hdfs_path, strict=False):
                hdfs_client.write(hdfs_path, data=csv_bytes, overwrite=True)
                logger.info(f"☁️ CSV sauvegardé immédiatement (HDFS) : {hdfs_path}")
        except Exception as e:
            logger.error(f"Erreur HDFS pour {num}: {e}")

    return True

In [ ]:
import os
import json
import logging
from datetime import datetime
import requests

# Configuration du logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - [%(levelname)s] - %(message)s"
)
logger = logging.getLogger(__name__)

# Headers de navigateur
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8"
}

PROGRESS_FILE = "progress_tracker.json"


# --- Fonction d'écriture en live sur disque ---
def save_progress_live(filepath: str, num_enterprise: str, status: str, details: dict = None):
    """Met à jour et force l'écriture immédiate du fichier JSON sur le disque (live write)."""
    data = {}
    
    # 1. Charger le fichier existant s'il existe
    if os.path.exists(filepath):
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
        except json.JSONDecodeError:
            data = {}

    # 2. Mettre à jour l'entrée
    data[num_enterprise] = {
        "status": status,
        "timestamp": datetime.now().isoformat(),
        "details": details or {}
    }

    # 3. Écriture atomique et synchronisée sur le disque
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
        f.flush()            # Vider le buffer Python
        os.fsync(f.fileno()) # Forcer l'écriture physique sur l'OS (Mac/Linux)

    logger.info(f"💾 Progression sauvegardée en live pour {num_enterprise} -> Status: {status}")


# --- Fonction HTTP isolée avec log ---
def make_http_request(url, session):
    try:
        response = session.get(url, headers=HEADERS, timeout=15)
        logger.info(
            f"HTTP Return | Status: {response.status_code} | "
            f"URL: {response.url} | Time: {response.elapsed.total_seconds():.2f}s"
        )
        return response
    except Exception as e:
        logger.error(f"HTTP Error | URL: {url} | Details: {e}")
        return None


def run_cbso_scraping(
    mongo_uri: str, 
    db_name: str, 
    excluded_forms: list[str], 
    tor_mgr: TorRotationManager,
    hdfs_client=None,
    limit: int = None,
    progress_file: str = "progress_tracker.json"
):
    """Pipeline final de scraping avec enregistrement streamé des CSV et suivi live."""
    client = MongoClient(mongo_uri)
    db = client[db_name]
    tracking_col = db["scraping_status"]
    
    tracking_col.create_index("enterpriseNumber", unique=True)

    # Récupération des entreprises cibles non exclues et non terminées
    pending_items = list(tracking_col.find({"status": {"$ne": "done"}}))
    logger.info(f"Total entreprises restant à traiter : {len(pending_items)}")

    processed_count = 0

    for item in pending_items:
        if limit is not None and processed_count >= limit:
            logger.info(f"🛑 Limite de {limit} entreprise(s) atteinte !")
            break

        num = item.get("enterpriseNumber") or str(item.get("EnterpriseNumber", "")).replace(".", "").strip()
        if not num:
            continue

        consecutive_429 = 0

        try:
            session = requests.Session()
            session.headers.update(HEADERS_BASE)
            session.proxies = tor_mgr.get_requests_proxies()
            
            referer_url = f"https://consult.cbso.nbb.be/consult-enterprise/{num}"
            session.headers["Referer"] = referer_url
            
            # 1. Initialisation de la session (cookies)
            resp_init = session.get(referer_url, timeout=10)
            if resp_init.status_code == 429:
                raise requests.exceptions.HTTPError("429 Too Many Requests", response=resp_init)
            resp_init.raise_for_status()

            # 2. Liste des dépôts disponibles
            deposits = get_published_deposits(session, num)
            csv_deposits = [d for d in deposits if d.get("periodEndDateYear", 0) >= 2021]
            
            downloaded_count = 0
            
            # 3. Boucle de téléchargement & stockage AU FUR ET À MESURE pour chaque année
            for dep in csv_deposits:
                dep_id = dep.get("id")
                year = dep.get("periodEndDateYear")
                
                if dep_id and year:
                    saved = download_and_store_csv_stream(
                        session=session,
                        enterprise_number=num,
                        dep_id=dep_id,
                        year=year,
                        hdfs_client=hdfs_client
                    )
                    if saved:
                        downloaded_count += 1
                    
                    # Pause légère pour étaler les requêtes
                    time.sleep(0.3)

            # 4. Succès : Mise à jour MongoDB + Live Tracking immédiats
            tracking_col.update_one(
                {"enterpriseNumber": num},
                {"$set": {
                    "status": "done",
                    "lastScrapedAt": datetime.utcnow(),
                    "documentsDownloaded": downloaded_count
                }}
            )

            save_progress_live(
                filepath=progress_file,
                num_enterprise=num,
                status="SUCCESS",
                details={"csv_count": downloaded_count}
            )

            processed_count += 1

        except requests.exceptions.HTTPError as err:
            status_code = err.response.status_code if err.response is not None else None
            logger.warning(f"HTTP Error {status_code} | Enterprise: {num}")
            
            if status_code == 429:
                handle_rate_limit(err.response, consecutive_429)
                consecutive_429 += 1
                tor_mgr.rotate() # Bascule de proxy Tor
            else:
                tracking_col.update_one(
                    {"enterpriseNumber": num},
                    {"$set": {"status": "error", "lastScrapedAt": datetime.utcnow()}}
                )
                save_progress_live(progress_file, num, f"HTTP_{status_code}")
                processed_count += 1
                
        except Exception as e:
            logger.error(f"Erreur globale sur l'entreprise {num}: {e}")
            tracking_col.update_one(
                {"enterpriseNumber": num},
                {"$set": {"status": "error", "lastScrapedAt": datetime.utcnow()}}
            )
            save_progress_live(progress_file, num, "ERROR", {"error_message": str(e)})
            processed_count += 1
# --- EXÉCUTION DU SCRIPT ---

# 1. Configuration Tor local
tor_proxies_config = [
    {"name": "tor1", "socks": "socks5h://127.0.0.1:9050", "control_host": "127.0.0.1", "control_port": 9051, "password": ""},
    {"name": "tor2", "socks": "socks5h://127.0.0.1:9052", "control_host": "127.0.0.1", "control_port": 9053, "password": ""},
    {"name": "tor3", "socks": "socks5h://127.0.0.1:9054", "control_host": "127.0.0.1", "control_port": 9055, "password": ""}
]

tor_manager = TorRotationManager(tor_proxies_config)

# 2. Configuration Mongo & Shunt rapide de l'étape de calcul (0 sec)
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "kbo_db"
formes_exclues = ["003", "007", "011", "012", "013", "016", "018", "020"]

# 3. Lancement direct du test sur 10 requêtes
logger.info("🧪 Démarrage du test limité à 10 entreprises avec sauvegarde live local...")

run_cbso_scraping(
    mongo_uri=MONGO_URI, 
    db_name=DB_NAME, 
    excluded_forms=formes_exclues, 
    tor_mgr=tor_manager,
    limit=10,
    progress_file=PROGRESS_FILE  # <--- Fichier local de suivi instantané
)

2026-07-29 09:21:14,167 - [INFO] - 🧪 Démarrage du test limité à 10 entreprises avec sauvegarde live local...
2026-07-29 09:21:19,519 - [INFO] - Total entreprises restant à traiter : 1461979
2026-07-29 09:21:24,477 - [WARNING] - HTTP Error 429 | Enterprise: 0200065765


[HTTP 429] Headers de la réponse : {'Date': 'Wed, 29 Jul 2026 07:21:24 GMT', 'Content-Type': 'text/html', 'Content-Length': '1484', 'Connection': 'close', 'Cache-Control': 'no-store', 'x-azure-ref': '20260729T072124Z-177769844dftr79jhC1CPHrgf80000000bx0000000002asw', 'X-Cache': 'CONFIG_NOCACHE'}
-> Retry-After absent : pause exponential backoff de 5s[cite: 1]


2026-07-29 09:21:29,492 - [INFO] - Error while receiving a control message (SocketClosed): empty socket content
2026-07-29 09:21:29,499 - [INFO] - Error while receiving a control message (SocketClosed): empty socket content


[Tor Alert] Connexion au port de contrôle échouée (127.0.0.1): socket connection failed (Received empty socket content.)
[Tor] Nouveau proxy actif : socks5h://127.0.0.1:9052[cite: 1]


2026-07-29 09:21:30,444 - [WARNING] - HTTP Error 429 | Enterprise: 0200068636


...

2026-07-29 09:22:05,316 - [INFO] - 💾 Progression sauvegardée en live pour 0201183146 -> Status: SUCCESS
2026-07-29 09:22:06,311 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/0201310731/cbso/csvs/2025.csv
2026-07-29 09:22:06,925 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/0201310731/cbso/csvs/2024.csv
2026-07-29 09:22:07,619 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/0201310731/cbso/csvs/2023.csv
2026-07-29 09:22:08,275 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/0201310731/cbso/csvs/2022.csv
2026-07-29 09:22:08,951 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/0201310731/cbso/csvs/2021.csv
2026-07-29 09:22:09,267 - [INFO] - 💾 Progression sauvegardée en live pour 0201310731 -> Status: SUCCESS
2026-07-29 09:22:10,492 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/0201310929/cbso/csvs/2025.csv
2026-07-29 09:22:11,257 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/020

[HTTP 429] Headers de la réponse : {'Date': 'Wed, 29 Jul 2026 07:22:18 GMT', 'Content-Type': 'text/html', 'Content-Length': '1484', 'Connection': 'close', 'Cache-Control': 'no-store', 'x-azure-ref': '20260729T072218Z-177769844dfr5fd6hC1CPHaqcw0000000awg00000000bp0s', 'X-Cache': 'CONFIG_NOCACHE'}
-> Retry-After absent : pause exponential backoff de 5s[cite: 1]


2026-07-29 09:22:23,285 - [INFO] - Error while receiving a control message (SocketClosed): empty socket content
2026-07-29 09:22:23,290 - [INFO] - Error while receiving a control message (SocketClosed): empty socket content


[Tor Alert] Connexion au port de contrôle échouée (127.0.0.1): socket connection failed (Received empty socket content.)
[Tor] Nouveau proxy actif : socks5h://127.0.0.1:9052[cite: 1]


2026-07-29 09:22:24,182 - [INFO] - 💾 Progression sauvegardée en live pour 0201339039 -> Status: SUCCESS
2026-07-29 09:22:25,710 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/0201400011/cbso/csvs/2025.csv
2026-07-29 09:22:26,709 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/0201400011/cbso/csvs/2024.csv
2026-07-29 09:22:27,652 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/0201400011/cbso/csvs/2023.csv
2026-07-29 09:22:28,675 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/0201400011/cbso/csvs/2022.csv
2026-07-29 09:22:29,708 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/0201400011/cbso/csvs/2021.csv
2026-07-29 09:22:30,013 - [INFO] - ⏩ Déjà téléchargé localement : ./data/raw/0201400011/cbso/csvs/2021.csv
2026-07-29 09:22:30,322 - [INFO] - 💾 Progression sauvegardée en live pour 0201400011 -> Status: SUCCESS
2026-07-29 09:22:32,028 - [INFO] - 💾 CSV sauvegardé immédiatement (Local) : ./data/raw/0201400110/cb

In [ ]:
import logging

# Configuration de base du logger si ce n'est pas déjà fait
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - [%(levelname)s] - %(message)s"
)
logger = logging.getLogger(__name__)

# --- Exemple de fonction HTTP avec log du retour HTTP ---
def make_http_request(url, session):
    try:
        response = session.get(url)
        # Log détaillé du retour HTTP
        logger.info(
            f"HTTP Return | Status: {response.status_code} | "
            f"URL: {response.url} | Time: {response.elapsed.total_seconds():.2f}s"
        )
        return response
    except Exception as e:
        logger.error(f"HTTP Error | URL: {url} | Details: {e}")
        return None


# Configuration des proxies Tor reliés aux ports exposés sur 127.0.0.1
tor_proxies_config = [
    {"socks": "socks5h://127.0.0.1:9050", "control_host": "127.0.0.1", "control_port": 9051, "password": ""},
    {"socks": "socks5h://127.0.0.1:9052", "control_host": "127.0.0.1", "control_port": 9053, "password": ""},
    {"socks": "socks5h://127.0.0.1:9054", "control_host": "127.0.0.1", "control_port": 9055, "password": ""}
]

# Initialisation du gestionnaire de rotation Tor
tor_manager = TorRotationManager(tor_proxies_config)

# Paramètres de connexion MongoDB
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "kbo_db"

# 2. Étape 9 : Identifier les formes juridiques à exclure
formes_exclues = find_excluded_juridical_forms(MONGO_URI, DB_NAME, sample_size=100)

# 3. Étape 10 : Lancer le scraping général avec suivi et stockage HDFS
run_cbso_scraping(
    mongo_uri=MONGO_URI, 
    db_name=DB_NAME, 
    excluded_forms=formes_exclues, 
    tor_mgr=tor_manager
)

Forme Juridique: 001 | Taux sans dépôt: 55.56%
Forme Juridique: 002 | Taux sans dépôt: 72.00%
Forme Juridique: 003 | Taux sans dépôt: 100.00%
Forme Juridique: 006 | Taux sans dépôt: 100.00%
Forme Juridique: 007 | Taux sans dépôt: 100.00%
Forme Juridique: 008 | Taux sans dépôt: 100.00%
Forme Juridique: 009 | Taux sans dépôt: 100.00%
Forme Juridique: 011 | Taux sans dépôt: 100.00%
Forme Juridique: 012 | Taux sans dépôt: 100.00%
Forme Juridique: 013 | Taux sans dépôt: 100.00%
Forme Juridique: 014 | Taux sans dépôt: 100.00%
Forme Juridique: 015 | Taux sans dépôt: 78.00%
Forme Juridique: 016 | Taux sans dépôt: 100.00%
Forme Juridique: 017 | Taux sans dépôt: 100.00%
Forme Juridique: 018 | Taux sans dépôt: 98.00%
Forme Juridique: 019 | Taux sans dépôt: 100.00%
Forme Juridique: 020 | Taux sans dépôt: 100.00%
Forme Juridique: 021 | Taux sans dépôt: 100.00%
Forme Juridique: 022 | Taux sans dépôt: 99.00%
Forme Juridique: 023 | Taux sans dépôt: 100.00%
Forme Juridique: 025 | Taux sans dépôt: 100.0

In [ ]:
from pymongo import MongoClient

# Connexion à MongoDB
client = MongoClient("mongodb://localhost:27017")
db = client["kbo_db"]

# Récupération d'un document exemple
document_exemple = db.entreprise.find_one()
print(document_exemple)

{'_id': ObjectId('6a676bc449d1f33a2397b64d'), 'EnterpriseNumber': '0200.065.765', 'Status': 'AC', 'JuridicalSituation': '000', 'TypeOfEnterprise': '2', 'JuridicalForm': '416', 'JuridicalFormCAC': '', 'StartDate': '09-08-1960', 'denominations': [{'_id': ObjectId('6a676bdf49d1f33a23cf784c'), 'EntityNumber': '0200.065.765', 'Language': '2', 'TypeOfDenomination': '001', 'Denomination': 'Intergemeentelijke Vereniging Veneco'}, {'_id': ObjectId('6a676bdf49d1f33a23cf784d'), 'EntityNumber': '0200.065.765', 'Language': '2', 'TypeOfDenomination': '002', 'Denomination': 'Veneco'}], 'addresses': [{'_id': ObjectId('6a676bf749d1f33a2302a63c'), 'EntityNumber': '0200.065.765', 'TypeOfAddress': 'REGO', 'CountryNL': '', 'CountryFR': '', 'Zipcode': '9070', 'MunicipalityNL': 'Destelbergen', 'MunicipalityFR': 'Destelbergen', 'StreetNL': 'Panhuisstraat', 'StreetFR': 'Panhuisstraat', 'HouseNumber': '1', 'Box': '', 'ExtraAddressInfo': '', 'DateStrikingOff': ''}], 'contacts': [], 'activities': [{'_id': ObjectI

In [ ]:
# Vérification des statuts de scraping
pipeline = [
    {"$group": {"_id": "$status", "count": {"$sum": 1}}}
]
stats = list(db.scraping_status.aggregate(pipeline))
print("--- Bilan par statut ---")
for st in stats:
    print(f"Statut {st['_id']} : {st['count']} entreprise(s)")

# Nombre total de fichiers téléchargés enregistrés
total_docs = list(db.scraping_status.aggregate([
    {"$group": {"_id": None, "total_csv": {"$sum": "$documentsDownloaded"}}}
]))
if total_docs:
    print(f"Total des fichiers CSV récupérés : {total_docs[0]['total_csv']}")

--- Bilan par statut ---
Statut error : 230797 entreprise(s)
Statut done : 19 entreprise(s)
Statut pending : 1231170 entreprise(s)
Total des fichiers CSV récupérés : 56
